In [ ]:
import os
from google.colab import files

# Install kaggle
!pip install -q kaggle

# Upload kaggle.json if not present
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download and unzip the dataset
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset
!mkdir -p /content/dataset
!unzip -q new-plant-diseases-dataset.zip -d /content/dataset

print("Dataset downloaded and extracted to /content/dataset")

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.70G/2.70G [01:09<00:00, 41.9MB/s]

Dataset downloaded and extracted to /content/dataset


# Task
Develop a plant disease classification system using the dataset at "/content/dataset". Implement data engineering with augmentation using ImageDataGenerator, train a baseline custom CNN, and perform transfer learning with ResNet50. Finally, evaluate and compare both models using classification reports and confusion matrices for the 38 disease categories.

## Data Engineering & Augmentation

### Subtask:
Set up directory paths and implement ImageDataGenerator for loading and augmenting the plant disease dataset.


**Reasoning**:
I will initialize the directory paths for the dataset and configure the ImageDataGenerators with augmentation for training and rescaling for validation and testing.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import csv
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,random_split
from PIL import Image
import torchvision import models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F
import kagglehub
# Download dataset
path =kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
train_base=os.path.join(path, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)")
train_dir=os.path.join(train_base, "train")
valid_dir=os.path.join(train_base, "valid")
transform=transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train=ImageFolder(train_dir, transform=transform)
valid_full=ImageFolder(valid_dir, transform=transform)
# Split validation into val and test
val_size=len(valid_full)//2
test_size=len(valid_full) - val_size
val_dataset,test_dataset=random_split(valid_full, [val_size, test_size])
# Define Loaders
train_loader=DataLoader(train, batch_size=32, shuffle=True)
val_loader=DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Data loaders initialized. Train size: {len(train)},Val size: {val_size},Test size: {test_size}")

Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
Data loaders initialized. Train size: 70295, Val size: 8786, Test size: 8786


In [ ]:
# Define the number of classes
num_classes = 38
# Load pre-trained EfficientNetB0
# weights=models.EfficientNet_B0_Weights.DEFAULT is the modern way to load pre-trained weights
model_efficientnet=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
# Freeze all parameters in the base network
for param in model_efficientnet.parameters():
    param.requires_grad=False
# Replace the classifier head
# EfficientNetB0 classifier is in model.classifier[1]
in_features=model_efficientnet.classifier[1].in_features
model_efficientnet.classifier=nn.Sequential(
    nn.Dropout(p=0.5, inplace=True),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(512, num_classes)
)

# Move model to device
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_efficientnet=model_efficientnet.to(device)
# Loss and Optimizer
criterion_eff=nn.CrossEntropyLoss()
optimizer_eff=optim.Adam(model_efficientnet.classifier.parameters(), lr=0.001)
print("EfficientNetB0 model defined and classifier initialized in PyTorch.")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 179MB/s]

EfficientNetB0 model defined and classifier initialized in PyTorch.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss=0.0
    correct=0
    total=0
    for images,labels in loader:
        images,labels=images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
        _,predicted=torch.max(outputs, 1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
    epoch_loss=running_loss/len(loader)
    epoch_acc=correct/total
    return epoch_loss,epoch_acc
def validate(model,loader,criterion,device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
# Training Parameters for EfficientNet
num_epochs_eff=10
scheduler_eff=torch.optim.lr_scheduler.StepLR(optimizer_eff, step_size=2, gamma=0.1)
best_val_acc_eff=0.0
history_eff = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
print(f"Starting EfficientNetB0 training on device: {device}")
for epoch in range(num_epochs_eff):
    # Reusing train_one_epoch and validate functions from the CNN training cell
    train_loss, train_acc=train_one_epoch(model_efficientnet, train_loader, criterion_eff, optimizer_eff, device)
    val_loss, val_acc =validate(model_efficientnet, val_loader, criterion_eff, device)

    scheduler_eff.step()

    if val_acc > best_val_acc_eff:
        best_val_acc_eff = val_acc
        torch.save(model_efficientnet.state_dict(), 'best_efficientnet.pth')
        print(f"*** New best EfficientNet model saved with val_acc: {val_acc:.4f} ***")

    history_eff['train_loss'].append(train_loss)
    history_eff['train_acc'].append(train_acc)
    history_eff['val_loss'].append(val_loss)
    history_eff['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs_eff} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"EfficientNet training complete. Best accuracy: {best_val_acc_eff:.4f}")

Starting EfficientNetB0 training on device: cuda
*** New best EfficientNet model saved with val_acc: 0.8792 ***
Epoch 1/10 | Train Loss: 0.9388 Acc: 0.7160 | Val Loss: 0.3822 Acc: 0.8792
*** New best EfficientNet model saved with val_acc: 0.8948 ***
Epoch 2/10 | Train Loss: 0.7924 Acc: 0.7551 | Val Loss: 0.3338 Acc: 0.8948
*** New best EfficientNet model saved with val_acc: 0.8968 ***
Epoch 3/10 | Train Loss: 0.7171 Acc: 0.7774 | Val Loss: 0.3272 Acc: 0.8968
*** New best EfficientNet model saved with val_acc: 0.9117 ***
Epoch 4/10 | Train Loss: 0.6772 Acc: 0.7865 | Val Loss: 0.2838 Acc: 0.9117
Epoch 5/10 | Train Loss: 0.6624 Acc: 0.7931 | Val Loss: 0.2904 Acc: 0.9097
Epoch 6/10 | Train Loss: 0.6576 Acc: 0.7952 | Val Loss: 0.3013 Acc: 0.9056
Epoch 7/10 | Train Loss: 0.6404 Acc: 0.7964 | Val Loss: 0.2981 Acc: 0.9044
Epoch 8/10 | Train Loss: 0.6510 Acc: 0.7962 | Val Loss: 0.3055 Acc: 0.9054
Epoch 9/10 | Train Loss: 0.6503 Acc: 0.7972 | Val Loss: 0.2875 Acc: 0.9101
Epoch 10/10 | Train Loss